# Week 5 Assignment: I Spy with my Little Eye 🕵️
### MI6 (SSS) vs Cheka (KGB) — Espionage Game Simulation

**Topics**: Bayesian Updates, Mixed Strategies, Imperfect Information, Game Theory

---

**Objective**: Simulate a strategic espionage game where MI6 uses mixed strategies (truth-telling and bluffing) while Cheka uses Bayesian reasoning to infer the true location of a classified file hidden in one of three safehouses.

**Mission Brief**: Two rival agencies compete to recover a classified file hidden in safehouse A, B, or C. MI6 gets noisy intelligence, decides what to tell Cheka (possibly lying), and where to actually search. Cheka uses Bayes' Rule to figure out where the file really is.

## Imports & Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
print('Setup complete ✅')

## Step 1 — Hide the File
Randomly place the classified file in one of three safehouses with equal probability: P(A) = P(B) = P(C) = 1/3.

In [ ]:
def hide_file():
    """Randomly place the classified file in one of three safehouses."""
    return np.random.choice(['A', 'B', 'C'])

## Step 2 — MI6 Receives Intelligence
The intelligence report is:
- **Correct** with probability `accuracy` (default 0.6)
- **Incorrect** with probability `1 - accuracy`, choosing uniformly among the remaining two locations

In [ ]:
def generate_intelligence(actual_location, accuracy=0.6):
    """
    MI6 receives an intelligence report.
    Correct with prob `accuracy`, else wrong location chosen uniformly.
    """
    if np.random.random() < accuracy:
        return actual_location
    else:
        remaining = [s for s in ['A', 'B', 'C'] if s != actual_location]
        return np.random.choice(remaining)

## Steps 3 & 4 — MI6's Mixed Strategy Decisions
MI6 makes **two independent decisions**:
1. **Message to Cheka**: Truth (prob `p_truth`) or bluff (prob `1 - p_truth`)
2. **Where MI6 actually searches**: Follow report (prob `p_follow`) or deviate

**Key insight**: These are independent — MI6 can lie to Cheka while secretly searching the correct location.

In [ ]:
def mi6_decisions(report, p_truth=0.6, p_follow=0.7):
    """
    MI6 decides what message to send and where to search.
    Uses mixed (randomized) strategy.
    
    Returns: (message, mi6_search_location, is_truthful)
    """
    remaining = [s for s in ['A', 'B', 'C'] if s != report]

    # Decision 1: Message to Cheka
    if np.random.random() < p_truth:
        message = report
        is_truthful = True
    else:
        message = np.random.choice(remaining)
        is_truthful = False

    # Decision 2: Where MI6 actually searches
    if np.random.random() < p_follow:
        mi6_search = report
    else:
        mi6_search = np.random.choice(remaining)

    return message, mi6_search, is_truthful

## Step 5 — Cheka's Bayesian Update (Core Math) 🧮

Cheka receives MI6's message `m` and computes the posterior:

$$P(\text{file at } s \mid \text{message} = m) \propto P(\text{message} = m \mid \text{file at } s) \times P(\text{file at } s)$$

The likelihood is computed by **marginalizing over all possible reports**:

$$P(m \mid s) = \sum_{r \in \{A,B,C\}} P(m \mid \text{report}=r) \times P(\text{report}=r \mid \text{file}=s)$$

Where:
- $P(\text{report}=r \mid \text{file}=s) = \text{accuracy}$ if $r=s$, else $(1-\text{accuracy})/2$
- $P(m \mid \text{report}=r) = p_{\text{truth}}$ if $m=r$, else $(1-p_{\text{truth}})/2$

In [ ]:
def bayes_update(prior, message, p_truth, intel_accuracy=0.6):
    """
    Cheka updates its belief using Bayes' Rule.
    Marginalizes over all possible reports MI6 could have received.
    
    Returns: posterior dict {safehouse: probability}
    """
    safehouses = ['A', 'B', 'C']
    posterior = {}
    
    for s in safehouses:
        p_s = prior[s]  # Prior
        
        # Compute P(message = m | file = s) by marginalizing over reports
        likelihood = 0.0
        for r in safehouses:
            # P(report = r | file = s)
            p_report = intel_accuracy if r == s else (1 - intel_accuracy) / 2.0
            # P(message = m | report = r)
            p_msg = p_truth if message == r else (1 - p_truth) / 2.0
            likelihood += p_msg * p_report
        
        posterior[s] = likelihood * p_s
    
    # Normalize
    total = sum(posterior.values())
    for s in safehouses:
        posterior[s] /= total if total > 0 else 1
    
    return posterior

# Quick test
prior = {'A': 1/3, 'B': 1/3, 'C': 1/3}
post = bayes_update(prior, 'A', p_truth=0.6, intel_accuracy=0.6)
print('Test — Prior: uniform, Message: A, p_truth: 0.6')
print(f'Posterior: A={post["A"]:.4f}, B={post["B"]:.4f}, C={post["C"]:.4f}')
print(f'Sum = {sum(post.values()):.4f} ✅' if abs(sum(post.values()) - 1.0) < 1e-9 else '❌')

## Steps 6 & 7 — Cheka's Search & Reward Matrix

| Outcome | MI6 | Cheka |
|---------|-----|-------|
| Only MI6 finds file | 10 | 0 |
| Only Cheka finds file | 0 | 10 |
| Both find file | 3 | 3 |
| Neither finds file | 0 | 0 |

In [ ]:
def cheka_search(posterior):
    """Cheka searches the safehouse with highest posterior probability."""
    return max(posterior, key=posterior.get)

def compute_rewards(actual, mi6_search, cheka_loc):
    """
    Award points based on who finds the file.
    Both prefer to find it alone (10) over sharing (3).
    """
    mi6_found = (mi6_search == actual)
    cheka_found = (cheka_loc == actual)
    if mi6_found and not cheka_found: return 10, 0
    elif cheka_found and not mi6_found: return 0, 10
    elif mi6_found and cheka_found: return 3, 3
    else: return 0, 0

## Step 8 — Full Simulation Engine

In [ ]:
def run_simulation(n_missions=1000, p_truth=0.6, p_follow=0.7, intel_accuracy=0.6):
    """
    Run the espionage game for n_missions and collect all statistics.
    """
    mi6_total = cheka_total = truthful_count = 0
    successful_bluffs = failed_bluffs = 0
    posterior_true_sum = 0.0

    for _ in range(n_missions):
        actual = hide_file()
        report = generate_intelligence(actual, accuracy=intel_accuracy)
        message, mi6_loc, is_truthful = mi6_decisions(report, p_truth, p_follow)

        prior = {'A': 1/3, 'B': 1/3, 'C': 1/3}
        posterior = bayes_update(prior, message, p_truth, intel_accuracy)
        posterior_true_sum += posterior[actual]

        cheka_loc = cheka_search(posterior)
        mi6_r, cheka_r = compute_rewards(actual, mi6_loc, cheka_loc)
        mi6_total += mi6_r
        cheka_total += cheka_r

        if is_truthful:
            truthful_count += 1
        else:
            if cheka_loc != actual: successful_bluffs += 1
            else: failed_bluffs += 1

    bluff_total = successful_bluffs + failed_bluffs
    return {
        'mi6_score': mi6_total, 'cheka_score': cheka_total,
        'truthful_messages': truthful_count,
        'successful_bluffs': successful_bluffs, 'failed_bluffs': failed_bluffs,
        'bluff_success_rate': successful_bluffs / max(1, bluff_total),
        'avg_posterior_true': posterior_true_sum / n_missions,
        'n_missions': n_missions
    }

def print_results(results, label=''):
    print(f'\n{"="*55}')
    if label: print(f'  {label}')
    print(f'{"="*55}')
    print(f'  Missions played      : {results["n_missions"]}')
    print(f'  MI6 Total Score      : {results["mi6_score"]}')
    print(f'  Cheka Total Score    : {results["cheka_score"]}')
    print(f'  Truthful Messages    : {results["truthful_messages"]}')
    print(f'  Successful Bluffs    : {results["successful_bluffs"]}')
    print(f'  Failed Bluffs        : {results["failed_bluffs"]}')
    print(f'  Bluff Success Rate   : {results["bluff_success_rate"]:.2%}')
    print(f'  Avg Posterior (True)  : {results["avg_posterior_true"]:.4f}')
    print(f'{"="*55}')

## Baseline Simulation (Default Parameters)

In [ ]:
np.random.seed(42)
baseline = run_simulation(n_missions=5000, p_truth=0.6, p_follow=0.7, intel_accuracy=0.6)
print_results(baseline, 'BASELINE: p_truth=0.6, p_follow=0.7, accuracy=0.6')

---
## Experiment 1: Varying p_truth
Vary `p_truth = {0, 0.25, 0.5, 0.75, 1.0}` while keeping `p_follow=0.7`, `accuracy=0.6`.

**Question**: How does always telling the truth compare with always bluffing?

In [ ]:
p_truth_values = [0.0, 0.25, 0.5, 0.75, 1.0]
mi6_scores_1, cheka_scores_1, bluff_rates_1 = [], [], []

for pt in p_truth_values:
    res = run_simulation(n_missions=5000, p_truth=pt, p_follow=0.7, intel_accuracy=0.6)
    print_results(res, f'p_truth = {pt}')
    mi6_scores_1.append(res['mi6_score'])
    cheka_scores_1.append(res['cheka_score'])
    bluff_rates_1.append(res['bluff_success_rate'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(p_truth_values, mi6_scores_1, 'o-', color='#2196F3', lw=2, ms=8, label='MI6')
axes[0].plot(p_truth_values, cheka_scores_1, 's-', color='#F44336', lw=2, ms=8, label='Cheka')
axes[0].set_xlabel('p_truth', fontsize=12); axes[0].set_ylabel('Total Score', fontsize=12)
axes[0].set_title('Experiment 1: Scores vs p_truth', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

axes[1].bar(p_truth_values, bluff_rates_1, width=0.12, color='#FF9800', edgecolor='black')
axes[1].set_xlabel('p_truth'); axes[1].set_ylabel('Bluff Success Rate')
axes[1].set_title('Experiment 1: Bluff Success Rate', fontweight='bold')
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

diff = [m - c for m, c in zip(mi6_scores_1, cheka_scores_1)]
colors = ['#4CAF50' if d > 0 else '#F44336' for d in diff]
axes[2].bar(p_truth_values, diff, width=0.12, color=colors, edgecolor='black')
axes[2].axhline(y=0, color='black', ls='--', lw=0.8)
axes[2].set_xlabel('p_truth'); axes[2].set_ylabel('MI6 − Cheka')
axes[2].set_title('Experiment 1: Score Advantage', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## Experiment 2: Varying p_follow
Vary `p_follow = {0.5, 0.7, 0.9, 1.0}` while keeping `p_truth=0.6`, `accuracy=0.6`.

**Question**: Does blindly following the report always lead to the highest score?

In [ ]:
p_follow_values = [0.5, 0.7, 0.9, 1.0]
mi6_scores_2, cheka_scores_2 = [], []

for pf in p_follow_values:
    res = run_simulation(n_missions=5000, p_truth=0.6, p_follow=pf, intel_accuracy=0.6)
    print_results(res, f'p_follow = {pf}')
    mi6_scores_2.append(res['mi6_score'])
    cheka_scores_2.append(res['cheka_score'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(p_follow_values)); w = 0.3
ax.bar(x - w/2, mi6_scores_2, w, label='MI6', color='#2196F3', edgecolor='black')
ax.bar(x + w/2, cheka_scores_2, w, label='Cheka', color='#F44336', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels([str(pf) for pf in p_follow_values])
ax.set_xlabel('p_follow'); ax.set_ylabel('Total Score')
ax.set_title('Experiment 2: Scores vs p_follow', fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

---
## Experiment 3: Varying Intelligence Accuracy
Test `accuracy = {0.4, 0.6, 0.8}` with full `p_truth` sweep.

**Question**: How does intel reliability affect the usefulness of bluffing?

In [ ]:
accuracies = [0.4, 0.6, 0.8]
p_truth_vals = [0.0, 0.25, 0.5, 0.75, 1.0]
results_all = {}

for acc in accuracies:
    mi6_s, cheka_s, bluff_r = [], [], []
    for pt in p_truth_vals:
        res = run_simulation(n_missions=5000, p_truth=pt, p_follow=0.7, intel_accuracy=acc)
        print_results(res, f'accuracy={acc}, p_truth={pt}')
        mi6_s.append(res['mi6_score'])
        cheka_s.append(res['cheka_score'])
        bluff_r.append(res['bluff_success_rate'])
    results_all[acc] = (mi6_s, cheka_s, bluff_r)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = {0.4: '#FF9800', 0.6: '#2196F3', 0.8: '#4CAF50'}

for acc in accuracies:
    mi6_s, cheka_s, bluff_r = results_all[acc]
    axes[0].plot(p_truth_vals, mi6_s, 'o-', color=cmap[acc], lw=2, ms=7, label=f'MI6 (acc={acc})')
    axes[1].plot(p_truth_vals, cheka_s, 's-', color=cmap[acc], lw=2, ms=7, label=f'Cheka (acc={acc})')
    axes[2].plot(p_truth_vals, bluff_r, '^-', color=cmap[acc], lw=2, ms=7, label=f'acc={acc}')

axes[0].set_xlabel('p_truth'); axes[0].set_ylabel('MI6 Total Score')
axes[0].set_title('Exp 3: MI6 Score by Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('p_truth'); axes[1].set_ylabel('Cheka Total Score')
axes[1].set_title('Exp 3: Cheka Score by Accuracy', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('p_truth'); axes[2].set_ylabel('Bluff Success Rate')
axes[2].set_title('Exp 3: Bluff Success Rate', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## Brief Questions — Answers

### Q1: How do Bayesian updates help Cheka?

If Cheka simply trusted every message, MI6 could always bluff and send Cheka to the wrong safehouse. **Bayesian updating** lets Cheka incorporate its knowledge of MI6's truthfulness probability into a rational belief distribution. When `p_truth` is low, the posterior actually *shifts away* from the reported location — Cheka learns to distrust. This is far superior to blind trust, which makes Cheka trivially exploitable.

### Q2: Do 'Always Truth' or 'Always Bluff' work well?

**Neither extreme works well:**
- **Always Truth (p_truth=1)**: Cheka learns MI6 is honest, always searches the same location → both share the 3-point reward instead of MI6 getting 10.
- **Always Bluff (p_truth=0)**: Cheka learns MI6 always lies, inverts the message → MI6's deception becomes self-defeating.

A **mixed strategy** (e.g. p_truth ≈ 0.3–0.5) keeps Cheka maximally uncertain. It cannot fully trust or distrust. This is the core insight of **mixed-strategy Nash Equilibria**: randomization prevents exploitation.

### Q3: Optimal MI6 Strategy?

Based on our simulations:
- **p_truth ≈ 0.25–0.5**: Bluff more than half the time. Keeps Cheka's posterior close to uniform (≈1/3 each), making its search nearly random.
- **p_follow ≈ 0.7–1.0**: Follow the intelligence report closely since it's 60% accurate. Deviating hurts MI6's own chances.

**Key insight**: MI6's *message strategy* (p_truth) and *search strategy* (p_follow) serve different purposes and should be optimized independently. The message is for deceiving Cheka; the search is for maximizing MI6's own find probability.